# How To Analyze Occupancy with Supervision

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow/supervision/blob/develop/docs/notebooks/occupancy_analytics.ipynb)
[![Roboflow](https://raw.githubusercontent.com/roboflow-ai/notebooks/main/assets/badges/roboflow-blogpost.svg)](https://blog.roboflow.com/occupancy-analytics/)

In this notebook, we'll use a parking lot to demonstrate how we can extract numerous informative metrics and detailed graphics, all from one video, using Supervision.

**This notebook accompanies the [Occupancy Analytics with Computer Vision](https://blog.roboflow.com/occupancy-analytics/) tutorial on the Roboflow Blog. Check it out for deeper explanations and context!**

![Example GIF](https://lh7-us.googleusercontent.com/VjbtO5CCQXpa3QzmxxDnb8PjGP7fLQivnwknHV6YUGcVOTqegZ9NiYf4c-jJvtopLQiBUYxchGoyru0jIlb0_pVLdh6_YjA5rB0rcDf7gnRH3NSJsJGUTKHO34qd5Bqkeo3Waq0LGuSUf6WYP79alfY)

In this notebook, we will cover the following:

1. Getting training data
2. Training a object detection model
3. Detect vehicles
4. Analyze data and generate statistics






## Before You Start
Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [ ]:
!nvidia-smi

## Install Relevant Packages
Here, we will install the Roboflow package, for uploading and training our model, and Supervision for visualization and extracting metrics from our predicted model results.

In [ ]:
!pip install roboflow supervision trackers -q

## Getting Video Data
We will start with turning a single video into a folder of frame images, for training our model. Upload your video and set your video's file path here.

In [ ]:
VIDEO_PATH = "/content/parkinglot1080.mov"

First, let's create a directory to save the video frames

In [ ]:
import os

FRAMES_DIR = "/content/frames"
os.mkdir(FRAMES_DIR)

Then, we can use Supervision's [`get_video_frames_generator` function](https://supervision.roboflow.com/latest/utils/video/#get_video_frames_generator) to get, then save, our video frames

In [ ]:
import supervision as sv
from PIL import Image

frames_generator = sv.get_video_frames_generator(VIDEO_PATH)

for i, frame in enumerate(frames_generator):
  img = Image.fromarray(frame)
  img.save(f"{FRAMES_DIR}/{i}.jpg")

print(f"Saved frames to {FRAMES_DIR}")

### Random Crop Sampling (If Using SAHI)
If we are using SAHI (which we are in our example), randomly sampling cropped portions of our image can help mimic the effect of SAHI detection during training, improving performance.

In [ ]:
# Note: This code block was written by ChatGPT

import os
import random
from PIL import Image
import numpy as np

# import shutil
# shutil.rmtree("augmented")

def random_crop(img):
    width, height = img.size

    crop_width = random.randint(int(width * 0.1), int(width * 0.4))
    crop_height = random.randint(int(height * 0.1), int(height * 0.4))

    left = random.randint(0, width - crop_width)
    top = random.randint(0, height - crop_height)

    return img.crop((left, top, left + crop_width, top + crop_height))

def augment_images(source_folder, target_folder, num_images=100):
    if not os.path.exists(target_folder):
        os.makedirs(target_folder)

    all_images = [file for file in os.listdir(source_folder) if file.endswith('.jpg')]

    selected_images = np.random.choice(all_images, size=min(num_images, len(all_images)), replace=False)

    for i, filename in enumerate(selected_images):
        with Image.open(os.path.join(source_folder, filename)) as img:
            cropped_img = random_crop(img)
            cropped_img.save(os.path.join(target_folder, f'augmented_{i}.jpg'))

# Paths to the source and target folders
source_folder = '/content/frames'
target_folder = '/content/augmented'

# Augment images
augment_images(source_folder, target_folder)


## Training a Model
Now that we have our images, we can upload our extracted frames as training data to Roboflow.

### Upload Training Data

In [ ]:
# Upload the extracted frames to Roboflow
import os
import roboflow

rf = roboflow.Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace().project("parking-lot-occupancy-detection-eoaek")

for filename in os.listdir(FRAMES_DIR):
  img_path = os.path.join(FRAMES_DIR, filename)
  if os.path.isfile(img_path):
      project.upload(image_path=img_path)

### Training Model Using Autodistill (Optional)
We can train our model using Automated Labeling, powered by Autodistill, to automatically label our data. Copy the code required for this section from the Roboflow app.

> Note: It's not required to use Autodistill

In [ ]:
# PASTE CODE FROM ROBOFLOW HERE

## Vehicle Detection
Now, we can run our model to get inference data for our video data.

### Setup Model
First, set the model up as a callback function so that we can call it later on while using Supervision.

In [ ]:
from roboflow import Roboflow
from trackers import ByteTrackTracker
import supervision as sv
import numpy as np
import cv2

rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY") # Get your own API key - This one won't work
project = rf.workspace().project("parking-lot-occupancy-detection-eoaek")
model = project.version("5").model

def callback(x: np.ndarray) -> sv.Detections:
    result = model.predict(x, confidence=25, overlap=30).json()
    return sv.Detections.from_inference(result)

### Configure Zones
Next, we will set up a list of the zones to be used with [PolygonZone](https://supervision.roboflow.com/latest/detection/tools/polygon_zone/). You can get these polygon coordinates using this [web utility](https://roboflow.github.io/polygonzone/).

For our example, we have have zones, but you can add as many or as little zones as you would like.

In [ ]:
# Polygons From PolygonZone

zones = [
    {
        'name': "Zone 1",
        'polygon': np.array([[229, 50],[-3, 306],[1, 614],[369, 50]]),
        'max': 32
    },
    {
        'name': 'Zone 2',
        'polygon': np.array([[465, 46],[177, 574],[401, 578],[609, 46]]),
        'max': 38
    },
    {
        'name': 'Zone 3',
        'polygon': np.array([[697, 58],[461, 858],[737, 858],[849, 58]]),
        'max': 46
    },
    {
        'name': 'Zone 4',
        'polygon': np.array([[941, 58],[909, 862],[1273, 858],[1137, 58]]),
        'max': 48
    },
    {
        'name': 'Zone 5',
        'polygon': np.array([[1229, 46],[1501, 1078],[1889, 1078],[1405, 46]]),
        'max': 52
    }
]

### Setup Supervision
For our use case, we will use the following features of Supervision. Refer to the linked documentation for more details:

*   [ByteTrack](https://trackers.roboflow.com/latest/), from the external `trackers` package: To track the location of our vehicles, so we can assess how long they are parked
*   [InferenceSlicer](https://supervision.roboflow.com/detection/tools/inference_slicer/?ref=blog.roboflow.com): A helper utility to run SAHI on our model
*   [TriangleAnnotator](https://supervision.roboflow.com/annotators/?ref=blog.roboflow.com#triangleannotator): To help visualize the locations of the vehicles
*   [HeatMapAnnotator](https://supervision.roboflow.com/annotators/?ref=blog.roboflow.com#heatmapannotator): To generate heatmaps so we can identify our busiest areas
*   [PolygonZone](https://supervision.roboflow.com/detection/tools/polygon_zone/?ref=blog.roboflow.com#polygonzone), [PolygonZoneAnnotator](https://supervision.roboflow.com/detection/tools/polygon_zone/?ref=blog.roboflow.com#polygonzoneannotator): To help count and identify vehicles in our respective zones and the annotator to help visualize those zones.

In [ ]:
# track_activation_threshold matches the confidence used by the model callback.
tracker = ByteTrackTracker(track_activation_threshold=0.25)
slicer = sv.InferenceSlicer(
    callback=callback,
    slice_wh=(800, 800),
    overlap_ratio_wh=(0.2, 0.2),
    thread_workers=10,
    iou_threshold=0.2
)
triangle_annotator = sv.TriangleAnnotator(
    base=20,
    height=20
)
heat_map_annotator = sv.HeatMapAnnotator()

def setup_zones(frame_wh):
  if zones:
    for zone in zones:
      zone['history'] = []
      zone['PolygonZone'] = sv.PolygonZone(
          polygon=zone['polygon']
      )
      zone['PolygonZoneAnnotator'] = sv.PolygonZoneAnnotator(
        zone=zone['PolygonZone'],
        color=sv.Color.WHITE,
        thickness=4,
    )

def process_frame(frame,heatmap=None):
    detections = slicer(image=frame)
    detections = tracker.update(detections)
    detections = detections[detections.tracker_id != -1]  # -1 = pending track

    annotated_frame = frame.copy()

    annotated_frame = triangle_annotator.annotate(
        scene=annotated_frame,
        detections=detections
    )

    if heatmap is None:
      heatmap = np.full(frame.shape, 255, dtype=np.uint8)

    heat_map_annotator.annotate(
      scene=heatmap,
      detections=detections
    )

    if zones:
      for zone in zones:
        zone_presence = zone['PolygonZone'].trigger(detections)
        zone_present_idxs = [idx for idx, present in enumerate(zone_presence) if present]
        zone_present = detections[zone_present_idxs]

        zone_count = len(zone_present)
        zone['history'].append(zone_count)


        annotated_frame = zone['PolygonZoneAnnotator'].annotate(
            scene=annotated_frame,
            label=f"{zone['name']}: {zone_count}"
        )

        # Heatmap
        heatmap = zone['PolygonZoneAnnotator'].annotate(
            scene=heatmap,
            label=" "
        )

    return annotated_frame, heatmap

### Try With a Single Image


In [ ]:
image = cv2.imread("./frames/5.jpg")
image_wh = (image.shape[1],image.shape[0])
setup_zones(image_wh)

annotated_image, heatmap = process_frame(image)

sv.plot_image(annotated_image)
sv.plot_image(heatmap)

### Setup Graphs
Before we run the model on the entire video, we will set up the logic to generate our graphs using matplotlib.


In [ ]:
# Credit to https://matplotlib.org/matplotblog/posts/matplotlib-cyberpunk-style/ for graph styles
%matplotlib agg
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO

def generate_graphs(max_frames):
  plt.ioff()
  # Plot Styles
  plt.style.use("seaborn-dark")
  for param in ['figure.facecolor', 'axes.facecolor', 'savefig.facecolor']:
      plt.rcParams[param] = '#212946'

  for param in ['text.color', 'axes.labelcolor', 'xtick.color', 'ytick.color']:
      plt.rcParams[param] = '0.9'


  dataframe = pd.DataFrame()
  graphs = {}


  for zone in zones:
    percentage_history = [(count/zone['max'])*100 for count in zone['history']]
    dataframe[zone['name']] = percentage_history
    plt.title(f'{zone["name"]} Usage')

    # Extra Styles
    fig, ax1 = plt.subplots()
    ax1.grid(color='#2A3459')

    # Data
    ax1.plot(zone["history"])

    # Axis Labeling
    plt.ylabel('Vehicles')
    plt.ylim(top=zone["max"])
    plt.xlim(right=max_frames)
    ax2 = ax1.twinx()
    ax2.set_ylabel('Occupied Percentage (%)')

    # Export Graph Image
    buf = BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
    buf.seek(0)
    graphs[zone['name']] = Image.open(buf)
    plt.close(fig)


  plt.ioff()
  dataframe.plot()

  # Axis
  plt.ylabel('Occupied (%)', fontsize=15)
  plt.ylim(top=100)
  plt.xlim(right=max_frames)

  # Export combined
  buf = BytesIO()
  plt.savefig(buf, format='png', bbox_inches='tight')
  buf.seek(0)

  plt.close()

  graphs['combined_percentage'] = Image.open(buf)

  return graphs

In [ ]:
generate_graphs(400)['combined_percentage']

### Process Video
Now, we can process the video to get detections from the entire video.

In [ ]:
VIDEO_PATH = "/content/parkinglot1080.mov"
MAIN_OUTPUT_PATH = "/content/parkinglot_annotated.mp4"
frames_generator = sv.get_video_frames_generator(source_path=VIDEO_PATH)
video_info = sv.VideoInfo.from_video_path(video_path=VIDEO_PATH)

setup_zones(video_info.resolution_wh)


with sv.VideoSink(target_path=MAIN_OUTPUT_PATH, video_info=video_info) as sink:
  heatmap = None
  for i, frame in enumerate(frames_generator):
    print(f"Processing frame {i}")

    # Infer
    annotated_frame, heatmap = process_frame(frame, heatmap)

    # Save the latest heatmap
    Image.fromarray(heatmap).save(f"/content/heatmap/{i}.jpg")

    # Create Graphs
    graphs = generate_graphs(video_info.total_frames)
    graph = graphs["combined_percentage"].convert("RGB")
    graph.save(f"/content/graphs/{i}.jpg")

    # sv.plot_image(annotated_frame)

    # Send as frame to video
    sink.write_frame(frame=annotated_frame)

### Generate Graphs/Heatmap Video (optional)

In [ ]:
import cv2
def create_videos_from_dir(dir,output):
  images = len(os.listdir(dir))-1

  sample_img_path = os.path.join(dir,f"1.jpg")
  sample_img = cv2.imread(sample_img_path)
  height, width, channels = sample_img.shape
  video_info = sv.VideoInfo(width=width,height=height,fps=24,total_frames=images)

  with sv.VideoSink(target_path=output, video_info=video_info) as sink:
    for i in range(images):
      path = os.path.join(dir,f"{i}.jpg")
      img = cv2.imread(path)
      sink.write_frame(frame=img)

# Graphs
create_videos_from_dir("/content/graphs","/content/parkinglot_graph.mp4")

# Heatmap
create_videos_from_dir("/content/heatmap","/content/parkinglot_heatmap.mp4")

## Analyze Data
Lastly, we can analyze the data we got to extract quantitative metrics from our video.

### Save your data for later
Using Pickle, we can save our zone detection data so that we can load it in for later analysis. Remember to download your file from the Colab file manager.

In [ ]:
import pickle

with open('parkinglot_zonedata.pkl', 'wb') as outp:
  pickle.dump(zones, outp, pickle.HIGHEST_PROTOCOL)

#### Import your data
To load your data back in, upload the saved file to the Colab environment and run the code cell.

In [ ]:
with open('parkinglot_zonedata.pkl', 'rb') as inp:
    zones_imported = pickle.load(inp)
    zones = zones_imported

### Occupancy Per Section
Since we recorded the number of objects (vehicles) in each zone, we can compare that against our hardcoded `max` that we put in while setting up our zones. Using this data, we can calculate the average and median occupancy, as well as any other metrics such as the max or the minimum occupancy throughout that time period.

In [ ]:
import statistics
for zone in zones:
    occupancy_percent_history = [(count/zone['max'])*100 for count in zone['history']]
    average_occupancy = round(statistics.mean(occupancy_percent_history))
    median_occupancy = round(statistics.median(occupancy_percent_history))
    highest_occupancy = round(max(occupancy_percent_history))
    lowest_occupancy = round(min(occupancy_percent_history))
    print(f"{zone['name']} had an average occupancy of {average_occupancy}% with a median occupancy of {median_occupancy}%.")

### Total Occupancy
Using the occupancy for the zones, we can also add up all the occupancy metrics throughout all the zones in order to calculate metrics for the whole parking lot.

In [ ]:
lot_history = []
for zone in zones:
    for idx, entry in enumerate(zone['history']):
      if(idx >= len(lot_history) or len(lot_history)==0): lot_history.append([])
      lot_history[idx].append(zone['history'][idx]/zone['max'])

lot_occupancy_history = [sum(entry)/len(entry)*100 for entry in lot_history]

average_occupancy = round(statistics.mean(lot_occupancy_history))
median_occupancy = round(statistics.median(lot_occupancy_history))
highest_occupancy = round(max(lot_occupancy_history))
lowest_occupancy = round(min(lot_occupancy_history))

print(f"The entire lot had an average occupancy of {average_occupancy}% with a median occupancy of {median_occupancy}%.")

In [ ]:
print(lot_occupancy_history)

# [
#    ...
#    73.51691310215338,
#    73.34063105087132,
#    73.86694684034501,
#    ...
# ]

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt

fig, ax1 = plt.subplots()
plt.title('Total Lot Usage')
ax1.grid(color='#2A3459')

ax1.plot(lot_occupancy_history)
ax1.set_ylabel('Occupied Percentage (%)')

plt.ylim(top=100)
plt.xlim(right=len(lot_occupancy_history))

plt.show()

### Busy Areas
Using Supervision's heat map annotator, we can use heatmaps while transforming the images in order to create images on top-down views of each zone.

In [ ]:
import cv2
import numpy as np

def transform_image(image, points):
    width = max(np.linalg.norm(points[0] - points[1]), np.linalg.norm(points[2] - points[3]))
    height = max(np.linalg.norm(points[0] - points[3]), np.linalg.norm(points[1] - points[2]))
    dest_points = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype="float32")
    matrix = cv2.getPerspectiveTransform(points.astype("float32"), dest_points)
    transformed_image = cv2.warpPerspective(image, matrix, (int(width), int(height)))

    return transformed_image

def generate_top_down_views(frame,show=True):
  heatmap = cv2.imread(f"heatmap/{frame}.jpg")
  image = cv2.imread(f"frames/{frame}.jpg")

  images = []

  for zone in zones:
    if show: print(f"Occupancy Visualization of {zone['name']}")
    top_down_image = transform_image(image, zone['polygon'])
    top_down_heatmap = transform_image(heatmap, zone['polygon'])

    combined_image = cv2.addWeighted(top_down_image, 0.7, top_down_heatmap, 0.3, 0)

    if show: sv.plot_image(combined_image, size=(5,5))

    images.append(combined_image)

  return images


In [ ]:
generate_top_down_views(400)

In [ ]:
import os
import numpy as np
from PIL import Image
import supervision as sv

for filename in os.listdir("frames"):
  img_path = os.path.join("frames", filename)
  heatmap_path = os.path.join("heatmap", filename)
  if os.path.isfile(img_path) and os.path.isfile(heatmap_path):
    frame = int(filename.replace(".jpg",""))
    images = generate_top_down_views(frame,False)
    gap = 10

    pil_images = [Image.fromarray(image) for image in images]

    # Resize images to have the same width
    widths, heights = zip(*(i.size for i in pil_images))
    max_width = max(widths)
    total_height = sum(heights) + gap * (len(images) - 1)
    resized_images = [i.resize((max_width, int(i.height * max_width / i.width))) for i in pil_images]

    # Create a new image with the correct combined size
    combined_image = Image.new('RGB', (max_width, total_height))

    # Paste each image into the combined image with the specified gap
    y_offset = 0
    for img in resized_images:
        combined_image.paste(img, (0, y_offset))
        y_offset += img.height + gap

    combined_image = combined_image.rotate(90, expand=True)

    combined_image.save(f"sectionheatmaps/{frame}.jpg")

    sv.plot_image(np.array(combined_image))